In [ ]:
import torch
import torchvision
import torch.nn as nn
import torchvision.transforms as transforms
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import zipfile

In [ ]:
transform = transforms.ToTensor()

with zipfile.ZipFile("C:\\Users\\Marien\\Downloads\\archive.zip", "r") as zip_ref:
    zip_ref.extractall("cats")

dataset = torchvision.datasets.ImageFolder(root="cats", transform=transform)

data_loader = DataLoader(dataset, batch_size=64, shuffle=True)



In [ ]:
class VAE(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size = 3, stride = 2, padding = 1),
            nn.ReLU(),

            nn.Conv2d(16, 32, kernel_size = 3, stride = 2, padding = 1),
            nn.ReLU(),

            nn.Conv2d(32, 64, kernel_size = 3, stride = 2, padding = 1),
            nn.ReLU(),

            nn.Flatten(),
        )

        self.mu = nn.Linear(64 * 8 * 8, 16)

        self.log_var = nn.Linear(64 * 8 * 8, 16)

        self.decoder = nn.Sequential(
            nn.Linear(16, 64 * 8 * 8),
            nn.ReLU(),
            nn.Unflatten(1, (64, 8, 8)),
            nn.ConvTranspose2d(64, 32, 3, stride = 2, padding = 1, output_padding = 1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 3, stride = 2, padding = 1, output_padding = 1),
            nn.ReLU(),
            nn.ConvTranspose2d(16, 3, 3, stride = 2, padding = 1, output_padding = 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.encoder(x)
        mu = self.mu(x)
        log_var = self.log_var(x)
        sigma = torch.exp(log_var/2)
        epsilon = torch.randn_like(sigma)
        z = mu + sigma * epsilon
        z = self.decoder(z)
        return z, mu, log_var

In [ ]:
reconstruction_loss = nn.L1Loss()

def KL_loss(log_var, mu):
    loss = 0
    for log_vari, mui in zip(log_var, mu):
        loss += -1/2 * torch.sum(1 + log_vari - mui.pow(2) - log_vari.exp())
    return loss/log_var.shape[0]

In [ ]:
model = VAE()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
#entrainement du modèle
for epoch in range(7):
    recc = 0
    kll = 0
    for x, _ in data_loader:
        x_hat, mu, log_var = model(x)
        rec = reconstruction_loss(x_hat, x)
        kl = KL_loss(log_var, mu) 
        loss = rec + 0.008 * kl
        optimizer.zero_grad()
        loss.backward()
        recc += rec.item()
        kll += kl.item()
        optimizer.step()
    print(f'Epoch [{epoch+1}/10], rec: {recc/len(data_loader):.4f}, kl: {kll/len(data_loader):.4f}')

In [ ]:
#Génération d'une image à partir d'un vecteur latent aléatoire
with torch.no_grad():
    z = torch.randn(1, 64)
    x = model.decoder(z)

plt.imshow(x[0].permute(1, 2, 0))
plt.show()

In [ ]:
#Test
with torch.no_grad():
    image = dataset[5000][0].unsqueeze(0)
    image.shape
    encode = model.encoder(image)
    mu = model.mu(encode)
    log_var = model.log_var(encode)
    z = mu 
    image_reconstructed = model.decoder(z)

plt.imshow(image_reconstructed[0].permute(1, 2, 0))
plt.show()
plt.imshow(image[0].permute(1, 2, 0))
plt.show()

print(mu, log_var)